#  Test data preparation.. 

**Business task:** Let's imagine that we have a bunch of ATMs or payment terminals and they constantly provide data in the form of JSON files, in batches. The application system must quickly process the files that have fallen into the folder and write their data to the transaction table. Then the application system must perform their application processing and form aggregated data.


Preparation of test data consists of:
1. Generating a given number of payment terminals and writing them to the created table.
2. Creating a given number of JSONL files with transactions for each terminal and writing them to files.
2. Gaining experience using the **Faker** package to generate personal data;
3. Creating DDL of postal tables via SPARK SQL;


## 1. Connecting the required python packages and preparatory operations

In [ ]:
from pathlib import Path
from notebookutils import mssparkutils
from faker import Faker
import pandas as pd 
import json
from pathlib import Path

location_pth="Files/bronze"
print(f"Create location path: {location_pth}")
folder_path = Path(location_pth)
folder_path.mkdir(parents=True, exist_ok=True)


## 2. Delete files if you need to delete them

In [ ]:
path = "Files/bronze"
# Get the list of files in the spacificed path
files = mssparkutils.fs.ls(path)

for f in files:
    mssparkutils.fs.rm(f.path, True)

print(f"All files in {path} have been deleted.")

## 3. Create a list  of payment terminals

### 3.1  Create the table of payment terminals

In [ ]:
# Create a table via spark.sql()
print(f"Create Table TERMINALS_D  Dictionary of pyment terminal")
spark.sql("DROP TABLE IF EXISTS TERMINALS_D")

spark.sql("""
        CREATE TABLE TERMINALS_D (
            TERMINAL_ID INT,
            TERMINAL_NAME STRING,
            ISACTIVE STRING
        )
        USING DELTA
"""
)

print(f"Table created")

### 3.2. Service function that adds a new terminal to the table or updates an existing one

In [ ]:
def add_Terminal(terminal_code, terminal_name):
    spark.sql("""
        MERGE INTO TERMINALS_D AS target
        USING (SELECT :TERMINAL_ID AS TERMINAL_ID, :TERMINAL_NAME AS TERMINAL_NAME, :ISACTIVE AS ISACTIVE) AS source
        ON target.TERMINAL_ID = source.TERMINAL_ID
        WHEN MATCHED THEN 
            UPDATE SET TERMINAL_NAME = source.TERMINAL_NAME
        WHEN NOT MATCHED THEN 
            INSERT (TERMINAL_ID, TERMINAL_NAME, ISACTIVE ) VALUES (source.TERMINAL_ID, source.TERMINAL_NAME, source.ISACTIVE)
    """, 
    args={"TERMINAL_ID": terminal_code, "TERMINAL_NAME": terminal_name, "ISACTIVE": "Y"}
    )


## 4. Generate transactions for terminals

In [ ]:


fake = Faker()

def make_record( terminal_code, opdt):
    """Generate operations for paricular file with fake data"""
    records=[]
    for _ in range(100):
        optype=fake.pyint(min_value=1, max_value= 5, step=1)
        amount=fake.pydecimal(left_digits=4, right_digits=2, positive=True, min_value=5.65,  max_value=1001.34)
        name = fake.name() 
     
        passport=fake.passport_number()
        proc = fake.pydecimal(left_digits=1, right_digits=2, positive=True, min_value=0.2,  max_value=2.5)
        # Генеруємо унікальний ID для кожної транзакції ВЖЕ ТУТ
        tx_id = f"TX-{terminal_code}-{fake.uuid4()[:8]}"
        record={
                "tx_id": tx_id, 
                "opdate": opdt.isoformat(),
                "terminal": terminal_code,
                "optype": optype,
                "name": name,
                "passport": passport,
                "amount": round(amount,2),
                "charge": round(amount*proc,2)  
                
        }  
        records.append(record) 
        
    return records

# generate file name and date for operations and store it into json
print("Preparation test data. Start.....")
for index in range(100):
    # terminal code
    terminal_code=fake.pyint(min_value=6300, max_value= 63200, step=1)
    terminal_name=fake.company()
    add_Terminal(terminal_code, terminal_name)
    # opdate
    opdt=fake.date_between(start_date='-1y')
    # json file name
    file_name=f"{terminal_code}_{opdt}.json"
    # paymant records
    records=make_record( terminal_code, opdt)
    output_path = f"/lakehouse/default/Files/bronze/{file_name}"
    # save data into json
    df=pd.DataFrame(records)
    df.to_json(output_path, orient='records', lines=True)
    print( f"{index}: Test Data saved into {output_path}")
    
print("Preparation test data. Finish")    

StatementMeta(, 4cd4c4c7-d8c3-44fe-8599-d6203634575d, 13, Finished, Available, Finished, False)

Preparation test data. Start.....
0: Test Data saved into /lakehouse/default/Files/bronze/42474_2025-07-10.json
1: Test Data saved into /lakehouse/default/Files/bronze/7354_2025-12-25.json
2: Test Data saved into /lakehouse/default/Files/bronze/21300_2026-04-06.json
3: Test Data saved into /lakehouse/default/Files/bronze/61546_2026-01-17.json
4: Test Data saved into /lakehouse/default/Files/bronze/49528_2026-02-08.json
5: Test Data saved into /lakehouse/default/Files/bronze/53865_2025-06-01.json
6: Test Data saved into /lakehouse/default/Files/bronze/60004_2025-08-04.json
7: Test Data saved into /lakehouse/default/Files/bronze/29129_2025-07-28.json
8: Test Data saved into /lakehouse/default/Files/bronze/19508_2025-11-22.json
9: Test Data saved into /lakehouse/default/Files/bronze/49338_2025-05-07.json
10: Test Data saved into /lakehouse/default/Files/bronze/51504_2025-06-10.json
11: Test Data saved into /lakehouse/default/Files/bronze/46789_2025-06-04.json
12: Test Data saved into /lak

## 5. View Terminal Dictionary 


In [ ]:
p_df=spark.sql("SELECT * FROM TERMINALS_D").toPandas()
p_df.head(40)

StatementMeta(, 9ba00502-a80d-4a05-b91d-e41a2566193c, 10, Finished, Available, Finished, False)

,TERMINAL_ID,TERMINAL_NAME,ISACTIVE
0,39134,"Rodriguez, Mullins and Hanson",Y
1,33763,"Hughes, Johnson and Mcdowell",Y
2,39520,"Ramirez, Johnson and Johnson",Y
3,32510,"Andrade, Green and Williams",Y
4,61937,"Jordan, Warren and Alvarado",Y
5,9252,"Ingram, Phillips and Fowler",Y
6,30104,"Wood, Johnson and Chandler",Y
7,51141,"Johnson, Jackson and Hill",Y
8,12616,"Johnson, Bennett and Long",Y
9,52064,"Roman, Pittman and Martin",Y
